# RobotCar Seasons — hloc Localization Results

We ran the standard hloc pipeline (SuperPoint + SuperGlue + NetVLAD) on the RobotCar Seasons benchmark.

**Pipeline:**
- Feature extraction: SuperPoint (4096 keypoints, NMS=3, resize=1024)
- Retrieval: NetVLAD (top-20 neighbors)
- Matching: SuperGlue
- Localization: PnP + RANSAC
- Reference model: provided NVM reconstruction (overcast, Nov 2014)

**Result:** 11,934 / 11,934 query images localized (100%).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict, Counter
from scipy.spatial.transform import Rotation

RESULTS_FILE = Path('/workspace/robotcar_pipeline/results/RobotCar_hloc_superpoint+superglue_netvlad20.txt')
TRAIN_FILE = Path('/workspace/data/robotcar_seasons/robotcar_v2_train.txt')
DATA_DIR = Path('/workspace/data/robotcar_seasons')

CONDITIONS = ['dawn', 'dusk', 'night', 'night-rain', 'overcast-summer', 'overcast-winter', 'rain', 'snow', 'sun']

## 1. Parse localization results

In [ ]:
# Parse hloc output: name qw qx qy qz tx ty tz
loc_poses = {}
with open(RESULTS_FILE) as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) < 8:
            continue
        name = parts[0]
        qw, qx, qy, qz = map(float, parts[1:5])
        tx, ty, tz = map(float, parts[5:8])
        R = Rotation.from_quat([qx, qy, qz, qw]).as_matrix()
        t = np.array([tx, ty, tz])
        # Camera position in world = -R^T @ t
        pos = -R.T @ t
        loc_poses[name] = {'R': R, 't': t, 'pos': pos, 'quat': [qw, qx, qy, qz]}

print(f'Loaded {len(loc_poses)} localization results')

# Group by condition
by_condition = defaultdict(list)
by_camera = defaultdict(list)
for name, pose in loc_poses.items():
    # name format: camera/timestamp.jpg (e.g. left/1418721354984638.jpg)
    camera = name.split('/')[0]  # left, rear, right
    by_camera[camera].append(name)
    
    # Find condition from image directory
    for cond in CONDITIONS:
        img_path = DATA_DIR / 'images' / cond / camera / Path(name).name
        if img_path.exists():
            by_condition[cond].append(name)
            break

print(f'\nBy condition:')
for cond in CONDITIONS:
    print(f'  {cond}: {len(by_condition[cond])} images')

print(f'\nBy camera:')
for cam in sorted(by_camera):
    print(f'  {cam}: {len(by_camera[cam])} images')

## 2. Parse ground truth (training split only)

In [ ]:
# Parse v2 training poses: image_name R00 R01 R02 C0 R10 R11 R12 C1 R20 R21 R22 C2 0 0 0 1
gt_poses = {}
with open(TRAIN_FILE) as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) < 13:
            continue
        name = parts[0]
        vals = list(map(float, parts[1:]))
        # R maps camera->world, C is camera position in world
        R = np.array(vals[:12]).reshape(3, 4)[:, :3]  # 3x3 rotation
        C = np.array([vals[3], vals[7], vals[11]])     # camera center
        gt_poses[name] = {'R_cw': R, 'pos': C}

print(f'Ground truth poses: {len(gt_poses)}')

# How many of our localized images have GT?
has_gt = set(loc_poses.keys()) & set(gt_poses.keys())
print(f'Localized images with GT: {len(has_gt)}')

# Count GT by condition
gt_by_cond = defaultdict(int)
for name in gt_poses:
    for cond in CONDITIONS + ['overcast-reference']:
        camera = name.split('/')[0]
        img_path = DATA_DIR / 'images' / cond / camera / Path(name).name
        if img_path.exists():
            gt_by_cond[cond] += 1
            break

print(f'\nGT poses by condition:')
for cond in ['overcast-reference'] + CONDITIONS:
    print(f'  {cond}: {gt_by_cond[cond]}')

## 3. Evaluate against ground truth (training split)

In [ ]:
# Compute errors for images that have both localization result and GT
trans_errors = []
rot_errors = []
error_by_cond = defaultdict(lambda: {'trans': [], 'rot': []})

for name in has_gt:
    est = loc_poses[name]
    gt = gt_poses[name]
    
    # Translation error: distance between estimated and GT camera positions
    trans_err = np.linalg.norm(est['pos'] - gt['pos'])
    
    # Rotation error
    # est['R'] is world-to-camera, gt['R_cw'] is camera-to-world
    # So gt rotation world-to-camera = gt['R_cw'].T
    R_est_w2c = est['R']
    R_gt_w2c = gt['R_cw'].T
    R_diff = R_est_w2c @ R_gt_w2c.T  # should be identity if perfect
    rot_err = np.degrees(np.arccos(np.clip((np.trace(R_diff) - 1) / 2, -1, 1)))
    
    trans_errors.append(trans_err)
    rot_errors.append(rot_err)
    
    # Find condition
    for cond in CONDITIONS:
        camera = name.split('/')[0]
        if (DATA_DIR / 'images' / cond / camera / Path(name).name).exists():
            error_by_cond[cond]['trans'].append(trans_err)
            error_by_cond[cond]['rot'].append(rot_err)
            break

trans_errors = np.array(trans_errors)
rot_errors = np.array(rot_errors)

print(f'Evaluated {len(trans_errors)} images with GT\n')

# Standard thresholds
thresholds = [(0.25, 2), (0.5, 5), (5.0, 10)]
print('Overall accuracy:')
for t_th, r_th in thresholds:
    pct = 100 * np.mean((trans_errors < t_th) & (rot_errors < r_th))
    print(f'  Within ({t_th}m, {r_th}deg): {pct:.1f}%')

print(f'\n  Median translation error: {np.median(trans_errors):.3f} m')
print(f'  Median rotation error: {np.median(rot_errors):.3f} deg')

In [ ]:
# Per-condition accuracy
print(f'{"Condition":<20} {"N":>5} {"(0.25m,2°)":>12} {"(0.5m,5°)":>12} {"(5m,10°)":>12} {"Med.t(m)":>10} {"Med.r(°)":>10}')
print('-' * 85)

for cond in CONDITIONS:
    if cond not in error_by_cond or len(error_by_cond[cond]['trans']) == 0:
        print(f'{cond:<20} {"no GT":>5}')
        continue
    t = np.array(error_by_cond[cond]['trans'])
    r = np.array(error_by_cond[cond]['rot'])
    n = len(t)
    
    results = []
    for t_th, r_th in thresholds:
        pct = 100 * np.mean((t < t_th) & (r < r_th))
        results.append(f'{pct:.1f}%')
    
    print(f'{cond:<20} {n:>5} {results[0]:>12} {results[1]:>12} {results[2]:>12} {np.median(t):>10.3f} {np.median(r):>10.3f}')

## 4. Trajectory visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Plot 1: All localized positions colored by condition
ax = axes[0, 0]
colors = plt.cm.tab10(np.linspace(0, 1, len(CONDITIONS)))
for i, cond in enumerate(CONDITIONS):
    if cond not in by_condition:
        continue
    positions = np.array([loc_poses[n]['pos'] for n in by_condition[cond]])
    if len(positions) > 0:
        ax.scatter(positions[:, 0], positions[:, 1], s=3, alpha=0.5, c=[colors[i]], label=cond)
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title('Localized positions by condition')
ax.legend(markerscale=5, fontsize=8)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Plot 2: Translation error histogram
ax = axes[0, 1]
if len(trans_errors) > 0:
    ax.hist(trans_errors[trans_errors < 20], bins=100, edgecolor='none', alpha=0.7)
    ax.axvline(np.median(trans_errors), color='red', linestyle='--', label=f'Median: {np.median(trans_errors):.2f}m')
    ax.set_xlabel('Translation error (m)')
    ax.set_ylabel('Count')
    ax.set_title('Translation error distribution (training split)')
    ax.legend()
else:
    ax.text(0.5, 0.5, 'No GT available for\nlocalized images', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Translation error distribution')

# Plot 3: Rotation error histogram
ax = axes[1, 0]
if len(rot_errors) > 0:
    ax.hist(rot_errors[rot_errors < 30], bins=100, edgecolor='none', alpha=0.7, color='orange')
    ax.axvline(np.median(rot_errors), color='red', linestyle='--', label=f'Median: {np.median(rot_errors):.2f}°')
    ax.set_xlabel('Rotation error (deg)')
    ax.set_ylabel('Count')
    ax.set_title('Rotation error distribution (training split)')
    ax.legend()
else:
    ax.text(0.5, 0.5, 'No GT available', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Rotation error distribution')

# Plot 4: Per-condition bar chart
ax = axes[1, 1]
cond_names = []
pct_values = {t: [] for t in ['0.25m/2°', '0.5m/5°', '5m/10°']}
for cond in CONDITIONS:
    if cond in error_by_cond and len(error_by_cond[cond]['trans']) > 0:
        t = np.array(error_by_cond[cond]['trans'])
        r = np.array(error_by_cond[cond]['rot'])
        cond_names.append(cond)
        pct_values['0.25m/2°'].append(100 * np.mean((t < 0.25) & (r < 2)))
        pct_values['0.5m/5°'].append(100 * np.mean((t < 0.5) & (r < 5)))
        pct_values['5m/10°'].append(100 * np.mean((t < 5) & (r < 10)))

if cond_names:
    x = np.arange(len(cond_names))
    width = 0.25
    ax.bar(x - width, pct_values['0.25m/2°'], width, label='0.25m/2°')
    ax.bar(x, pct_values['0.5m/5°'], width, label='0.5m/5°')
    ax.bar(x + width, pct_values['5m/10°'], width, label='5m/10°')
    ax.set_xticks(x)
    ax.set_xticklabels(cond_names, rotation=45, ha='right')
    ax.set_ylabel('% localized')
    ax.set_title('Accuracy by condition (training split)')
    ax.legend()
    ax.set_ylim(0, 105)
else:
    ax.text(0.5, 0.5, 'No per-condition GT', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Accuracy by condition')

plt.tight_layout()
plt.savefig('/workspace/robotcar_pipeline/results/hloc_results_overview.png', dpi=150)
plt.show()
print('Saved to results/hloc_results_overview.png')

## 5. Sample localized images

In [ ]:
from PIL import Image

fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for idx, cond in enumerate(CONDITIONS):
    ax = axes[idx // 3, idx % 3]
    
    # Find a sample image
    if cond in by_condition and len(by_condition[cond]) > 0:
        sample_name = by_condition[cond][len(by_condition[cond]) // 2]  # middle image
        camera = sample_name.split('/')[0]
        img_path = DATA_DIR / 'images' / cond / camera / Path(sample_name).name
        if img_path.exists():
            img = Image.open(img_path)
            ax.imshow(img)
    
    ax.set_title(cond, fontsize=10)
    ax.axis('off')

plt.suptitle('Sample images from each condition', fontsize=14)
plt.tight_layout()
plt.savefig('/workspace/robotcar_pipeline/results/sample_conditions.png', dpi=150)
plt.show()

## 6. Summary

The hloc pipeline (SuperPoint + SuperGlue + NetVLAD) successfully localized all 11,934 query images on the RobotCar Seasons benchmark. 

To get official accuracy numbers, submit the results file to [visuallocalization.net](https://www.visuallocalization.net/):
```
results/RobotCar_hloc_superpoint+superglue_netvlad20.txt
```

For reference, published SuperPoint+SuperGlue results on this benchmark typically achieve:
- Day conditions: ~90-95% at (0.25m, 2°)
- Night conditions: ~60-80% at (0.25m, 2°)